# GridPin — offline geocoding in five minutes

[GridPin](https://gridpin.dev) geocodes addresses from **one local file per country**: no API key, no rate limit,
and your queries never leave the machine running this notebook. The only network step is downloading a country
sheet once, checked against the signed release.

This notebook installs GridPin, downloads the Serbia sheet (32 MB, the smallest), and runs forward, reverse and
batch geocoding. Change `COUNTRY` to try another country.

In [ ]:
!pip install -q gridpin gridpin-mcp

## 1. Download a country sheet

Countries: `br` Brazil, `de` Germany (without Bavaria), `es` Spain, `fr` France, `it` Italy, `nl` Netherlands,
`rs` Serbia. Sizes range from 32 MB (Serbia) to 677 MB (Brazil).

In [ ]:
import gridpin
from gridpin_mcp import sheets

COUNTRY = "rs"
for s in sheets.SHEETS.values():
    print(f"{s.code}  {s.name:<12} {s.size / 1e6:6.0f} MB")

path = sheets.download(COUNTRY)  # verified against the signed release; a no-op once downloaded
geocoder = gridpin.Geocoder(str(path))
print("sheet:", path)

## 2. Address → coordinates

Each candidate carries `precision`: `house` is a door-level match, `street` and `city` are approximate.
GridPin returns nothing rather than a guess.

In [ ]:
def show(hits):
    for h in hits:
        print(f"{h['precision']:<6} {h['lat']:.6f}, {h['lon']:.6f}  {h['street']} {h.get('housenumber', '')}, "
              f"{h['commune']}  (confidence {h['confidence']:.2f})")
    if not hits:
        print("no match")

show(geocoder.geocode("Knez Mihailova 5, Beograd", 3))

## 3. Coordinates → nearest address

Reverse geocoding is approximate: streets are indexed by centroid, so each result carries `distance_m`.

In [ ]:
for h in geocoder.reverse(44.8176, 20.4569, 3):
    print(f"{h['distance_m']:6.1f} m  {h['street']} {h.get('housenumber', '')}, {h['commune']}")

## 4. A batch of addresses

`geocode_many` spreads a list across all cores. Paste your own addresses, or read a CSV column with pandas.

In [ ]:
import time
import pandas as pd

addresses = [
    "Knez Mihailova 5, Beograd",
    "Bulevar oslobođenja 10, Novi Sad",
    "Obrenovićeva 1, Niš",
    "Kneza Miloša 20, Beograd",
]
start = time.perf_counter()
results = geocoder.geocode_many(addresses, 1)
elapsed = time.perf_counter() - start

table = pd.DataFrame([
    {"query": q, **({k: hits[0].get(k) for k in ("precision", "lat", "lon", "street", "housenumber", "commune")}
                     if hits else {"precision": None})}
    for q, hits in zip(addresses, results)
])
print(f"{len(addresses)} addresses in {elapsed * 1000:.1f} ms")
table

## Next

- Install locally: `pip install gridpin gridpin-mcp`, then `gridpin-fetch rs` — the same sheet, on your laptop.
- For AI agents: an MCP server, `uvx gridpin-mcp` ([guide](https://github.com/gridpin/gridpin/blob/main/mcp/README.md)).
- Docs: https://gridpin.dev/docs · Source: https://github.com/gridpin/gridpin
- Data: Overture Maps and national registries — see
  [attributions](https://github.com/gridpin/gridpin/blob/main/ATTRIBUTIONS.en.md).